<a href="https://colab.research.google.com/github/arif202536599-cloud/gene_ids_matching/blob/main/Rice_TBtools_Gene_Matcher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Rice gene/protein matching after TBtools

Run each cell from top to bottom. Use a CPU runtime; no GPU or Google Drive mount is needed.

This notebook matches **exact model IDs**, follows GFF3 `Parent` links to genes, checks coordinates, and calculates sequence properties. Gene symbols come from curated annotation, not a machine-learning prediction. Training ML cannot establish an exact genomic location or an official gene symbol from an unlabelled protein list.

Upload these three files separately when prompted:

| Prompt | Your file |
|---|---|
| Selection file | `Final_matched_genes.txt`, an ID list, CSV/TSV, or protein FASTA |
| Matching annotation | `osa1_r7.all_models.gff3` (or `.gff3.gz`) |
| Protein FASTA | `final_protein_ids.txt` (FASTA content, with `>` headers) |

Use files from the same organism and annotation release. This workflow is designed for your MSU release 7 japonica rice export, and also accepts one ID per line, FASTA headers, or a CSV/TSV with an ID column. Coordinate tables without headers must begin with chromosome, model ID, start, end, and strand. The file labelled CDS by TBtools may contain transcript spans; actual CDS sizes will be taken from the original GFF3 CDS features.

Outputs: an Excel-readable CSV, a styled HTML/PDF table, a matching audit, duplicate FASTA report, and amino-acid features for a future labelled ML study. A ZIP downloads at the end.


## 1. Prepare the runtime

In [1]:
import importlib.util, subprocess, sys
for package in ['pandas', 'reportlab']:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
from pathlib import Path
from google.colab import files
from IPython.display import display, HTML
import datetime, hashlib, urllib.request
BASE = Path('/content/rice_gene_matching')
INPUT = BASE / 'inputs'
OUTPUT = BASE / 'results'
INPUT.mkdir(parents=True, exist_ok=True)
def upload_one(label):
    print(label)
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Select exactly ONE file for this prompt, then rerun this cell.')
    filename, content = next(iter(uploaded.items()))
    path = INPUT / Path(filename).name
    path.write_bytes(content)
    print('Loaded:', path.name, '| bytes:', len(content))
    return path


## 2. Upload your selection file
Choose your TBtools matched table, a plain ID list, a CSV/TSV with a model/gene ID header, or a protein FASTA. The format is detected automatically.

Model IDs match exactly. Gene-level IDs without coordinates expand to all annotated transcript models. If you provide only IDs or FASTA headers, model coordinates are taken from the GFF3 and no input-coordinate comparison is reported.


In [2]:
TB_PATH = upload_one('Upload the matched table, ID list, or protein FASTA selecting your models:')


Upload the matched table, ID list, or protein FASTA selecting your models:


Saving Osj_pep.fas to Osj_pep.fas
Loaded: Osj_pep.fas | bytes: 1163843


## 3. Upload the original GFF3
Select **osa1_r7.all_models.gff3** only. Do not upload the protein FASTA at this step.

In [3]:
GFF_PATH = upload_one('Upload the matching GFF3 annotation:')


Upload the matching GFF3 annotation:


Saving osa1_r7.all_models.gff3 to osa1_r7.all_models.gff3
Loaded: osa1_r7.all_models.gff3 | bytes: 81498659


## 4. Upload the protein FASTA
Select **final_protein_ids.txt**. A `.txt` extension is fine if its contents are FASTA. Protein model IDs must match the GFF3 model IDs.

In [4]:
PROTEIN_PATH = upload_one('Upload the protein FASTA:')


Upload the protein FASTA:


Saving final_protein_ids.txt to final_protein_ids.txt
Loaded: final_protein_ids.txt | bytes: 39894


## 5. Load the matching and calculation functions

Molecular weight uses average amino-acid masses and subtracts water for peptide bonds. Theoretical pI is the zero-charge root using Bjellqvist pKa parameters, including sequence-specific terminal corrections. Values describe an unmodified protein.

Calculation references: [Biopython IUPACData](https://github.com/biopython/biopython/blob/master/Bio/Data/IUPACData.py) and [IsoelectricPoint](https://github.com/biopython/biopython/blob/master/Bio/SeqUtils/IsoelectricPoint.py). No neural-network prediction is involved.


In [5]:
import csv, gzip, hashlib, io, json, re
from collections import Counter, defaultdict
from pathlib import Path
from urllib.parse import unquote
import pandas as pd

AA = 'ACDEFGHIKLMNPQRSTVWY'
MASSES = dict(zip(AA, [89.0932,121.1582,133.1027,147.1293,165.1891,75.0666,155.1546,131.1729,146.1876,131.1729,149.2113,132.1179,115.1305,146.1445,174.201,105.0926,119.1192,117.1463,204.2252,181.1885]))

def open_text(path):
    return gzip.open(path, 'rt', encoding='utf-8-sig') if str(path).lower().endswith('.gz') else open(path, encoding='utf-8-sig')

def read_tbtools(path):
    """Read coordinate tables, headered CSV/TSV, plain IDs, or FASTA headers."""
    rows = []; header = None; fasta = False
    norm = lambda value: re.sub('[^a-z0-9]', '', value.lower())
    id_headers = ('proteinmodelid','modelid','proteinid','transcriptid','id','geneid')
    aliases = {'chromosome':('chromosome','chr','chrno'),
               'start':('start','startbp','modelstartbp'),
               'end':('end','endbp','modelendbp'),
               'strand':('strand',), 'parent':('parent','parentgeneid','genelocus')}
    def append_record(line_no, key, values, format_name):
        if not key: raise ValueError(f'Input line {line_no}: missing model/gene ID.')
        values = {**dict(chromosome=None,start=None,end=None,strand=None,parent=''),**values}
        for field in ('start','end'):
            value = values[field]
            if value is not None and str(value).strip() not in {'','-','NA','N/A','nan'}:
                try:
                    number = float(value)
                    if not number.is_integer(): raise ValueError()
                    values[field] = int(number)
                except ValueError:
                    raise ValueError(f'Input line {line_no}: {field} must be an integer coordinate.')
            else: values[field] = None
        if values['start'] is not None and values['start'] < 1: raise ValueError(f'Input line {line_no}: start must be at least 1.')
        if values['end'] is not None and values['start'] is not None and values['end'] < values['start']: raise ValueError(f'Input line {line_no}: end precedes start.')
        if values['strand'] in {'','-','+'}:
            values['strand'] = values['strand'] or None
        elif values['strand'] is not None: raise ValueError(f'Input line {line_no}: strand must be + or -.')
        rows.append(dict(line=line_no,model=key,requested_id=key,input_format=format_name,**values))
    with open_text(path) as f:
        for line_no, line in enumerate(f, 1):
            if not line.strip() or line.startswith('#'): continue
            text = line.strip()
            if text.startswith('>'):
                if rows and not fasta: raise ValueError('The input mixes a table/ID list with FASTA records.')
                fasta = True
                label = text[1:].strip()
                ids = set(re.findall(r'\bLOC_Os\d{2}g\d+(?:\.\d+)?\b',label))
                if len(ids)>1: raise ValueError(f'FASTA header on line {line_no} contains multiple locus IDs.')
                key = next(iter(ids)) if ids else (label.split()[0] if label else '')
                append_record(line_no,key,{},'FASTA headers')
                continue
            if fasta:
                if not re.fullmatch('[A-Za-z* .-]+',text): raise ValueError(f'Unexpected FASTA sequence text on line {line_no}.')
                continue  # Only headers select IDs; protein measurements use PROTEIN_PATH.
            delimiter = '\t' if '\t' in text else (',' if ',' in text else None)
            fields = next(csv.reader([text],delimiter=delimiter)) if delimiter else text.split()
            if delimiter is None and norm(text) in id_headers: fields = [text]
            fields = [value.strip() for value in fields]
            columns = [norm(value) for value in fields]
            if len(fields)==9 and fields[2] in {'gene','mRNA','transcript','CDS','exon','five_prime_UTR','three_prime_UTR'} and fields[3].isdigit() and fields[4].isdigit():
                raise ValueError('A GFF3 was uploaded as the selection file. Rerun the TBtools/ID-selection upload with your matched table, ID list, or protein FASTA. Upload the annotation at the GFF3 prompt.')
            coordinate_row = len(fields)>=5 and fields[2].isdigit() and fields[4] in {'+','-'}
            if not rows and header is None and not coordinate_row and any(value in id_headers for value in columns):
                id_col = next(columns.index(value) for value in id_headers if value in columns)
                header = {'id':id_col}
                for field,names in aliases.items():
                    index = next((columns.index(value) for value in names if value in columns),None)
                    if index is not None: header[field]=index
                if 'geneid' in columns and columns.index('geneid')!=id_col: header['parent']=columns.index('geneid')
                continue
            if header is not None:
                if len(fields)<=max(header.values()): raise ValueError(f'Input line {line_no}: fewer columns than the header.')
                append_record(line_no,fields[header['id']],{k:fields[i] for k,i in header.items() if k!='id'},'Headered table')
            elif len(fields)==1 and re.fullmatch(r'LOC_Os\d{2}g\d+(?:\.\d+)?',fields[0]):
                append_record(line_no,fields[0],{},'ID list')
            elif len(fields)>=5:
                parents = [fields[i+1] for i,v in enumerate(fields[:-1]) if v=='Parent']
                if len(set(parents))>1: raise ValueError(f'Conflicting Parent values on line {line_no}.')
                append_record(line_no,fields[1],dict(chromosome=fields[0],start=fields[2],end=fields[3],strand=fields[4],parent=parents[0] if parents else ''),'Coordinate table')
            else:
                raise ValueError(f'Cannot recognize input line {line_no}: {text[:100]!r}. Upload a TBtools coordinate table, a CSV/TSV with a model/gene ID header, one LOC_Os ID per line, or FASTA headers. If this is the GFF3, upload it at the GFF3 prompt instead.')
    if not rows: raise ValueError('The selection file contains no usable IDs.')
    print('Selection format:', ', '.join(sorted({r['input_format'] for r in rows})), '| input records:',len(rows))
    return rows

def read_gff(path):
    genes, models, segments = {}, {}, defaultdict(set)
    with open_text(path) as f:
        for line_no, line in enumerate(f, 1):
            if line.startswith('##FASTA'): break
            if line.startswith('#') or not line.strip(): continue
            v = line.rstrip('\r\n').split('\t')
            if len(v) != 9: raise ValueError(f'GFF3 line {line_no}: expected nine columns.')
            if v[2] not in {'gene','mRNA','transcript','CDS'}: continue
            a = {k:unquote(value) for k,value in (s.split('=',1) for s in v[8].split(';') if '=' in s)}
            feature = dict(chromosome=v[0],start=int(v[3]),end=int(v[4]),strand=v[6],parent=a.get('Parent',''),attrs=a)
            if v[2] == 'CDS':
                for parent in a.get('Parent','').split(','):
                    if parent: segments[parent].add((int(v[3]),int(v[4])))
            else:
                key = a.get('ID')
                if not key: continue
                target = genes if v[2] == 'gene' else models
                if key in target and target[key] != feature:
                    raise ValueError(f'GFF3 contains conflicting records for ID {key}.')
                target[key] = feature
    return genes, models, segments

def read_proteins(path):
    groups = defaultdict(list); header = None; seq = []
    def finish():
        if header is None: return
        ids = set(re.findall(r'\bLOC_Os\d{2}g\d+(?:\.\d+)?\b', header))
        if len(ids) > 1: raise ValueError(f'Ambiguous FASTA header: {header}')
        key = next(iter(ids)) if ids else header.split()[0]
        groups[key].append(''.join(seq).upper().removesuffix('*'))
    with open_text(path) as f:
        for line in f:
            if line.startswith('>'):
                finish(); header=line[1:].strip(); seq=[]
                if not header: raise ValueError('Empty FASTA header.')
            elif line.strip():
                if header is None: raise ValueError('Protein FASTA must start with > followed by an ID.')
                seq.append(''.join(line.split()))
    finish()
    if not groups: raise ValueError('No proteins found in FASTA.')
    proteins, duplicate_rows = {}, []
    for key, values in groups.items():
        unique = set(values)
        proteins[key] = next(iter(unique)) if len(unique)==1 else None
        if len(values)>1: duplicate_rows.append({'Protein/model ID':key,'FASTA records':len(values),'Identical sequences':len(unique)==1})
    return proteins, duplicate_rows

def protein_metrics(seq):
    if not seq: raise ValueError('Empty protein sequence.')
    unknown = set(seq)-set(AA)
    if unknown: raise ValueError('Nonstandard/ambiguous residues: '+','.join(sorted(unknown)))
    mw = (sum(MASSES[a] for a in seq)-(len(seq)-1)*18.0153)/1000
    counts = Counter(seq); counts.update({'Nterm':1,'Cterm':1})
    positive = {'Nterm':{'A':7.59,'M':7.0,'S':6.93,'P':8.36,'T':6.82,'V':7.44,'E':7.7}.get(seq[0],7.5),'K':10.0,'R':12.0,'H':5.98}
    negative = {'Cterm':{'D':4.55,'E':4.75}.get(seq[-1],3.55),'D':4.05,'E':4.45,'C':9.0,'Y':10.0}
    def charge(ph):
        return sum(counts[a]/(1+10**(ph-pk)) for a,pk in positive.items())-sum(counts[a]/(1+10**(pk-ph)) for a,pk in negative.items())
    low, high=0.0,14.0
    for _ in range(60):
        mid=(low+high)/2
        if charge(mid)>0: low=mid
        else: high=mid
    return mw,(low+high)/2

def read_names(text):
    """Require named columns. Never guess gene symbols from descriptions."""
    rows=list(csv.reader(io.StringIO(text.lstrip('\ufeff')),delimiter='\t'))
    norm=lambda s:re.sub('[^a-z0-9]','',s.lower())
    for h,row in enumerate(rows[:30]):
        columns=[norm(s) for s in row]
        msu=next((i for i,c in enumerate(columns) if c in {'msuid','msuids','msulocusid','msuidentifier'}),None)
        symbol=next((i for i,c in enumerate(columns) if c in {'cgsnlgenesymbol','genesymbol','symbol'}),None)
        if msu is not None and symbol is not None: break
    else:
        raise ValueError('Name file must have tab-separated MSU ID and CGSNL Gene Symbol columns. Download OryzabaseGeneListEn from the linked page.')
    name=next((i for i,c in enumerate(columns) if c in {'cgsnlgenename','genename'}),None)
    trait=next((i for i,c in enumerate(columns) if c in {'traitid','traitgeneid','oryzabaseid'}),None)
    mapping=defaultdict(set)
    for row in rows[h+1:]:
        if len(row)<=max(msu,symbol): continue
        sym=row[symbol].strip()
        if sym in {'','-','NA','N/A'}: continue
        full=row[name].strip() if name is not None and len(row)>name else ''
        source=row[trait].strip() if trait is not None and len(row)>trait else ''
        for gene in set(re.findall(r'LOC_Os\d{2}g\d+',row[msu])):
            mapping[gene].add((sym,full,source))
    return mapping

def make_tables(tb_path,gff_path,protein_path,names=None):
    tb=read_tbtools(tb_path); genes,models,cds=read_gff(gff_path); proteins,dups=read_proteins(protein_path)
    by_gene=defaultdict(list)
    for key,feature in models.items(): by_gene[feature['parent']].append(key)
    expanded=[]
    for record in tb:
        has_coords=any(record[k] is not None for k in ('chromosome','start','end','strand'))
        if record['model'] in genes and record['model'] not in models and not has_coords and by_gene.get(record['model']):
            expanded.extend({**record,'model':key,'selection_note':'Gene-level ID expanded to all annotated isoforms'} for key in by_gene[record['model']])
        else: expanded.append(record)
    tb=expanded
    names=names or {}; counts=Counter(r['model'] for r in tb)
    output=[]; audits=[]; feature_rows=[]; seen=set()
    for serial,r in enumerate(tb,1):
        key=r['model']; model=models.get(key); issues=[]
        if counts[key]>1: issues.append('Repeated model ID in TBtools; rows preserved')
        if model is None: issues.append('Model ID not found in GFF3')
        parent=model['parent'] if model else r['parent']
        if ',' in parent: issues.append('Multiple gene parents; unresolved'); parent=''
        gene=genes.get(parent)
        if model and not gene: issues.append('Parent gene not found in GFF3')
        supplied=[k for k in ('chromosome','start','end','strand') if r[k] is not None]
        coord_ok=all(r[k]==model[k] for k in supplied) if model and supplied else None
        if model and coord_ok is False: issues.append('TBtools/GFF3 coordinate conflict; table uses GFF3')
        if model and r['parent'] and r['parent']!=parent: issues.append('TBtools/GFF3 Parent conflict')
        segments=sorted(cds.get(key,set())); overlap=any(segments[i][1]>=segments[i+1][0] for i in range(len(segments)-1))
        cds_bp=sum(end-start+1 for start,end in segments) if model and segments and not overlap else None
        if model and not segments: issues.append('No CDS segments')
        if overlap: issues.append('Overlapping CDS segments; length withheld')
        seq=proteins.get(key); aa=len(seq) if seq else None; mw=pi=None
        if key not in proteins: issues.append('Protein ID not found in FASTA')
        elif seq is None: issues.append('Conflicting FASTA sequences for same ID')
        elif not seq: issues.append('Empty protein sequence')
        else:
            try: mw,pi=protein_metrics(seq)
            except ValueError as e: issues.append(str(e))
        delta=cds_bp-3*aa if cds_bp is not None and aa is not None else None
        if delta is not None and delta not in {0,3}: issues.append('CDS/protein length disagreement; inspect annotation')
        matches=names.get(parent,set()); symbols=sorted({x[0] for x in matches}); full_names=sorted({x[1] for x in matches if x[1]})
        name_status='single curated symbol' if len(symbols)==1 else ('multiple curated symbols; review' if symbols else 'not found in available name source')
        symbol=symbols[0] if len(symbols)==1 else None
        annotation=model['attrs'] if model else {}
        if not symbol and not symbols and gene:
            symbol=gene['attrs'].get('gene_name') or gene['attrs'].get('gene')
            if symbol: name_status='GFF3 gene symbol; verify annotation evidence'
        if len(symbols)>1: issues.append('Multiple curated gene symbols; name withheld')
        start=model['start'] if model else None; end=model['end'] if model else None
        output.append({'Sr. #':serial,'Gene ID':parent or None,'Protein/model ID':key,'Common name':symbol,'Chr. No.':model['chromosome'] if model else None,'Strand':model['strand'] if model else None,'Model start (bp)':start,'Model end (bp)':end,'Model size (bp)':end-start+1 if model else None,'Gene size (bp)':gene['end']-gene['start']+1 if gene else None,'CDS size (bp)':cds_bp,'Protein size (AA)':aa,'Mol. wt. (kDa)':mw,'pI':pi})
        audits.append({'Sr. #':serial,'Input line':r['line'],'Requested ID':r['requested_id'],'Input format':r['input_format'],'Selection note':r.get('selection_note',''),'Protein/model ID':key,'Gene ID':parent,'ID match':model is not None,'Coordinates match':coord_ok if model else None,'Input start':r['start'],'Input end':r['end'],'Input Parent':r['parent'],'Gene start':gene['start'] if gene else None,'Gene end':gene['end'] if gene else None,'Name status':name_status,'All curated symbols':'; '.join(symbols),'Full gene names':'; '.join(full_names),'Oryzabase record IDs':'; '.join(sorted({x[2] for x in matches if x[2]})),'Product description':gene['attrs'].get('Note','') if gene else annotation.get('Note',''),'CDS segments':'; '.join(f'{s}-{e}' for s,e in segments),'CDS minus 3xAA':delta,'Issues':'; '.join(issues)})
        if seq and mw is not None and key not in seen:
            seen.add(key)
            feature={'Gene ID':parent,'Protein/model ID':key,'Common name':symbol,'Length':aa,'MW_kDa':mw,'pI':pi,'Sequence SHA256':hashlib.sha256(seq.encode()).hexdigest()}
            feature.update({f'AA_fraction_{a}':seq.count(a)/aa for a in AA}); feature_rows.append(feature)
    return pd.DataFrame(output),pd.DataFrame(audits),pd.DataFrame(feature_rows),pd.DataFrame(dups,columns=['Protein/model ID','FASTA records','Identical sequences'])

def export_results(table,audit,features,duplicates,output_dir,manifest):
    from html import escape
    import shutil
    out=Path(output_dir); out.mkdir(parents=True,exist_ok=True)
    for name,frame in [('matched_gene_table',table),('matching_audit',audit),('protein_features_for_ML',features),('duplicate_fasta_ids',duplicates)]:
        frame.to_csv(out/f'{name}.csv',index=False,encoding='utf-8-sig',float_format='%.6f')
    notes=['Rows follow the TBtools input order; isoforms and repeated input rows are retained.',
        'Table coordinates come from exact GFF3 model IDs. Coordinate conflicts are recorded in matching_audit.csv.',
        'Model and gene sizes are inclusive genomic spans (end - start + 1), including introns. CDS size is summed from distinct, nonoverlapping CDS segments.',
        'Protein lengths exclude one terminal stop symbol. MW is theoretical average mass for an unmodified linear protein. pI is the zero-charge root using Bjellqvist parameters.',
        'A dash means unavailable or withheld because the sources conflict. Common names are gene symbols, not inferred descriptions. Missing names do not imply unnamed genes.',
        'CDS/protein length agreement is a length check only. Translation identity requires the matching genome or CDS sequence.',
        'Features are provided for a future labelled ML study. This notebook verifies IDs; it does not train an ML model or establish gene function.']
    source_lines=[f'{key}: {value}' for key,value in manifest.items()]
    display_table=table.astype(object).where(pd.notna(table),'-')
    html_table=display_table.to_html(index=False,na_rep='-',escape=True,float_format=lambda x:f'{x:.2f}')
    html='<!doctype html><meta charset="utf-8"><title>Rice gene and protein table</title><style>body{font-family:Georgia,"Times New Roman",serif;margin:24px;color:#000}table{border-collapse:collapse;font-size:12px;width:100%}thead{border-top:1px solid;border-bottom:1px solid}th,td{padding:6px;text-align:center;border:none}tbody{border-bottom:1px solid}p{font-size:12px} @media print{@page{size:A3 landscape;margin:12mm}thead{display:table-header-group}tr{break-inside:avoid}}</style><h2>Matched gene and protein models in japonica rice</h2>'+html_table+''.join('<p>'+escape(x)+'</p>' for x in notes+source_lines)
    (out/'styled_gene_table.html').write_text(html,encoding='utf-8')
    (out/'methods_and_sources.txt').write_text('\n'.join(notes+['']+source_lines),encoding='utf-8')
    (out/'input_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
    # Printable table with no vertical rules, repeated headers, and embedded serif font when available.
    from reportlab.platypus import SimpleDocTemplate,Table,TableStyle,Paragraph,Spacer
    from reportlab.lib.pagesizes import A3,landscape
    from reportlab.lib.styles import ParagraphStyle
    from reportlab.pdfbase import pdfmetrics
    from reportlab.pdfbase.ttfonts import TTFont
    font='Times-Roman'; bold='Times-Bold'
    for directory in ('/usr/share/fonts/truetype/dejavu','/usr/share/fonts/truetype/liberation2'):
        normal=Path(directory)/('DejaVuSerif.ttf' if 'dejavu' in directory else 'LiberationSerif-Regular.ttf')
        heavy=Path(directory)/('DejaVuSerif-Bold.ttf' if 'dejavu' in directory else 'LiberationSerif-Bold.ttf')
        if normal.exists() and heavy.exists():
            pdfmetrics.registerFont(TTFont('GeneSerif',str(normal)));pdfmetrics.registerFont(TTFont('GeneSerifBold',str(heavy)));font='GeneSerif';bold='GeneSerifBold';break
    head_style=ParagraphStyle('header',fontName=bold,fontSize=8,leading=10,alignment=1)
    cell_style=ParagraphStyle('cell',fontName=font,fontSize=8.5,leading=10,alignment=1)
    note_style=ParagraphStyle('note',fontName=font,fontSize=8,leading=11,spaceAfter=4)
    title_style=ParagraphStyle('title',fontName=bold,fontSize=13,leading=16)
    def formatted(value,col):
        if pd.isna(value): return '-'
        if col in {'pI','Mol. wt. (kDa)'}: return f'{value:.2f}'
        return str(int(value)) if isinstance(value,(int,float)) else str(value)
    grid=[[Paragraph(escape(c).replace(' (','<br/>('),head_style) for c in table.columns]]
    grid += [[Paragraph(escape(formatted(value,col)),cell_style) for col,value in row.items()] for _,row in table.iterrows()]
    widths=[30,112,125,76,60,48,76,76,82,82,78,83,82,46]
    pdf_table=Table(grid,colWidths=widths,repeatRows=1,hAlign='LEFT')
    pdf_table.setStyle(TableStyle([('VALIGN',(0,0),(-1,-1),'MIDDLE'),('TOPPADDING',(0,0),(-1,-1),3),('BOTTOMPADDING',(0,0),(-1,-1),3),('LINEABOVE',(0,0),(-1,0),0.6,'black'),('LINEBELOW',(0,0),(-1,0),0.6,'black'),('LINEBELOW',(0,-1),(-1,-1),0.6,'black')]))
    elements=[Paragraph('Matched gene and protein models in japonica rice',title_style),Spacer(1,8),pdf_table,Spacer(1,10)]
    elements += [Paragraph(escape(x),note_style) for x in notes]
    doc=SimpleDocTemplate(str(out/'styled_gene_table.pdf'),pagesize=landscape(A3),leftMargin=35,rightMargin=35,topMargin=35,bottomMargin=35)
    doc.build(elements)
    return Path(shutil.make_archive(str(out.parent/'Rice_gene_matching_results'),'zip',out))


## 6. Obtain common names from Oryzabase

The default cell downloads **OryzabaseGeneListEn** from the [official download page](https://shigen.nig.ac.jp/rice/oryzabase/download/gene). It matches the gene locus to the list's MSU ID and CGSNL Gene Symbol fields. Isoform suffixes in the name reference are grouped at the gene level; isoforms in your results remain separate.

If the site cannot be reached or its format changes, matching continues with names marked unavailable. You can download the English list manually, change `UPLOAD_NAMES_MANUALLY` to `True`, rerun this cell, and then rerun steps 7 and 8. Multiple curated symbols for one locus are reported for review rather than choosing one silently.


In [6]:
UPLOAD_NAMES_MANUALLY = False  # Set True only to upload an official gene list yourself.
NAME_URL = 'https://shigen.nig.ac.jp/rice/oryzabase/gene/download?classtag=GENE_EN_LIST'
NAME_PATH = None
NAME_MAP = {}
NAME_STATUS = 'unavailable'
try:
    if UPLOAD_NAMES_MANUALLY:
        NAME_PATH = upload_one('Upload OryzabaseGeneListEn:')
    else:
        request = urllib.request.Request(NAME_URL, headers={'User-Agent':'RiceGeneMatching/1.0'})
        with urllib.request.urlopen(request, timeout=60) as response:
            content = response.read()
        NAME_PATH = INPUT / 'OryzabaseGeneListEn.txt'
        NAME_PATH.write_bytes(content)
    NAME_MAP = read_names(NAME_PATH.read_text(encoding='utf-8-sig'))
    NAME_STATUS = 'Oryzabase gene list loaded'
    print(NAME_STATUS, '| loci with symbols:', len(NAME_MAP))
except Exception as exc:
    print('Common-name lookup unavailable:', str(exc))
    print('Exact ID matching and protein measurements can still run. Missing names will remain blank.')


Common-name lookup unavailable: <urlopen error [Errno 111] Connection refused>
Exact ID matching and protein measurements can still run. Missing names will remain blank.


## 7. Match and review

This step retains input order. A gene-level selection without coordinates expands to all annotated isoforms, and the requested input ID is recorded in the audit. A shared parent gene is an isoform relationship, not automatically an accidental duplicate. Exact duplicate IDs in the FASTA are audited; conflicting sequences are not silently selected.

The table uses GFF3 model coordinates. Read the audit if TBtools coordinates or Parent values disagree. CDS length is summed from distinct, nonoverlapping CDS segments. A CDS length of three times the protein length plus a stop codon is a useful length check, not proof that the translated sequences are identical.


In [7]:
TABLE, AUDIT, FEATURES, FASTA_DUPLICATES = make_tables(TB_PATH, GFF_PATH, PROTEIN_PATH, NAME_MAP)
for col in ['Sr. #','Model start (bp)','Model end (bp)','Model size (bp)','Gene size (bp)','CDS size (bp)','Protein size (AA)']:
    TABLE[col] = TABLE[col].astype('Int64')
print('Selected input records:', AUDIT['Input line'].nunique())
print('Output model rows:', len(TABLE))
print('Matched models:', int(AUDIT['ID match'].sum()), 'of', len(TABLE))
print('Unique matched gene loci:', TABLE.loc[AUDIT['ID match'], 'Gene ID'].nunique())
print('Rows with a common name:', int(TABLE['Common name'].notna().sum()))
print('Coordinate conflicts:', int((AUDIT['Coordinates match'] == False).sum()))
print('Rows without input coordinates:', int(AUDIT['Coordinates match'].isna().sum()))
problems = AUDIT[AUDIT['Issues'].str.len() > 0]
print('Rows needing review:', len(problems))
display(TABLE.head(12))
if len(problems):
    display(problems[['Sr. #','Protein/model ID','Issues']])
if len(FASTA_DUPLICATES): display(FASTA_DUPLICATES)


Selection format: FASTA headers | input records: 2408
Selected input records: 2408
Output model rows: 2408
Matched models: 2408 of 2408
Unique matched gene loci: 1862
Rows with a common name: 0
Coordinate conflicts: 0
Rows without input coordinates: 2408
Rows needing review: 2343


,Sr. #,Gene ID,Protein/model ID,Common name,Chr. No.,Strand,Model start (bp),Model end (bp),Model size (bp),Gene size (bp),CDS size (bp),Protein size (AA),Mol. wt. (kDa),pI
0,1,LOC_Os02g58440,LOC_Os02g58440.1,None,Chr2,+,35738982,35743320,4339,4339,1944,<NA>,NaN,NaN
1,2,LOC_Os06g03670,LOC_Os06g03670.1,None,Chr6,+,1434731,1435857,1127,1127,645,<NA>,NaN,NaN
2,3,LOC_Os02g10760,LOC_Os02g10760.1,None,Chr2,+,5688698,5689806,1109,1109,618,<NA>,NaN,NaN
3,4,LOC_Os02g04340,LOC_Os02g04340.1,None,Chr2,-,1913425,1916183,2759,2759,2010,<NA>,NaN,NaN
4,5,LOC_Os12g37970,LOC_Os12g37970.1,None,Chr12,-,23325526,23327055,1530,1530,729,<NA>,NaN,NaN
5,6,LOC_Os04g47890,LOC_Os04g47890.1,None,Chr4,+,28470386,28474272,3887,3887,921,<NA>,NaN,NaN
6,7,LOC_Os05g41070,LOC_Os05g41070.1,None,Chr5,+,24062649,24066043,3395,3395,1008,<NA>,NaN,NaN
7,8,LOC_Os02g07840,LOC_Os02g07840.1,None,Chr2,+,4109770,4112884,3115,3115,837,<NA>,NaN,NaN
8,9,LOC_Os01g07480,LOC_Os01g07480.1,None,Chr1,-,3551044,3553146,2103,2103,756,<NA>,NaN,NaN
9,10,LOC_Os02g39540,LOC_Os02g39540.2,None,Chr2,+,23859837,23864637,4801,4801,2457,<NA>,NaN,NaN


,Sr. #,Protein/model ID,Issues
0,1,LOC_Os02g58440.1,Protein ID not found in FASTA
1,2,LOC_Os06g03670.1,Protein ID not found in FASTA
2,3,LOC_Os02g10760.1,Protein ID not found in FASTA
3,4,LOC_Os02g04340.1,Protein ID not found in FASTA
4,5,LOC_Os12g37970.1,Protein ID not found in FASTA
...,...,...,...
2403,2404,LOC_Os01g51140.2,Protein ID not found in FASTA
2404,2405,LOC_Os04g40140.2,Protein ID not found in FASTA
2405,2406,LOC_Os11g16290.2,Protein ID not found in FASTA
2406,2407,LOC_Os03g49880.1,Protein ID not found in FASTA


## 8. Export and download

Open `matched_gene_table.csv` in Excel. Open `styled_gene_table.html` in a browser or use the PDF for the serif table with horizontal rules.

`matching_audit.csv` contains full gene names, product descriptions, gene coordinates, name sources and match conflicts. `protein_features_for_ML.csv` has protein lengths, pI, molecular weight, amino-acid fractions and gene IDs. It is a feature dataset, not an ML result. A future classification study needs meaningful verified labels (for example, experimentally established family membership), independent validation, and grouping by gene/homology to avoid isoform leakage. Gene symbols themselves should continue to be looked up, not predicted.


In [8]:
def file_info(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(1024*1024), b''): digest.update(chunk)
    return {'filename': path.name, 'bytes': path.stat().st_size, 'sha256': digest.hexdigest()}
MANIFEST = {
    'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'TBtools': file_info(TB_PATH), 'GFF3': file_info(GFF_PATH), 'protein_FASTA': file_info(PROTEIN_PATH),
    'name_source_status': NAME_STATUS, 'name_source_URL': NAME_URL,
    'name_file': file_info(NAME_PATH) if NAME_PATH and NAME_PATH.exists() else None,
    'method': 'Exact ID/Parent matching; theoretical sequence properties; no ML training',
    'pI_reference': 'https://github.com/biopython/biopython/blob/master/Bio/SeqUtils/IsoelectricPoint.py',
    'mass_reference': 'https://github.com/biopython/biopython/blob/master/Bio/Data/IUPACData.py',
    'rows': len(TABLE), 'matched_rows': int(AUDIT['ID match'].sum())
}
ZIP_PATH = export_results(TABLE, AUDIT, FEATURES, FASTA_DUPLICATES, OUTPUT, MANIFEST)
print('Saved outputs:')
for path in sorted(OUTPUT.iterdir()): print(' ', path.name)
display(HTML((OUTPUT / 'styled_gene_table.html').read_text(encoding='utf-8')))
files.download(str(ZIP_PATH))


Saved outputs:
  duplicate_fasta_ids.csv
  input_manifest.json
  matched_gene_table.csv
  matching_audit.csv
  methods_and_sources.txt
  protein_features_for_ML.csv
  styled_gene_table.html
  styled_gene_table.pdf


Sr. #,Gene ID,Protein/model ID,Common name,Chr. No.,Strand,Model start (bp),Model end (bp),Model size (bp),Gene size (bp),CDS size (bp),Protein size (AA),Mol. wt. (kDa),pI
1,LOC_Os02g58440,LOC_Os02g58440.1,-,Chr2,+,35738982,35743320,4339,4339,1944,-,-,-
2,LOC_Os06g03670,LOC_Os06g03670.1,-,Chr6,+,1434731,1435857,1127,1127,645,-,-,-
3,LOC_Os02g10760,LOC_Os02g10760.1,-,Chr2,+,5688698,5689806,1109,1109,618,-,-,-
4,LOC_Os02g04340,LOC_Os02g04340.1,-,Chr2,-,1913425,1916183,2759,2759,2010,-,-,-
5,LOC_Os12g37970,LOC_Os12g37970.1,-,Chr12,-,23325526,23327055,1530,1530,729,-,-,-
6,LOC_Os04g47890,LOC_Os04g47890.1,-,Chr4,+,28470386,28474272,3887,3887,921,-,-,-
7,LOC_Os05g41070,LOC_Os05g41070.1,-,Chr5,+,24062649,24066043,3395,3395,1008,-,-,-
8,LOC_Os02g07840,LOC_Os02g07840.1,-,Chr2,+,4109770,4112884,3115,3115,837,-,-,-
9,LOC_Os01g07480,LOC_Os01g07480.1,-,Chr1,-,3551044,3553146,2103,2103,756,-,-,-
10,LOC_Os02g39540,LOC_Os02g39540.2,-,Chr2,+,23859837,23864637,4801,4801,2457,-,-,-


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Understanding the columns

| Field | Interpretation |
|---|---|
| Gene ID | Parent gene locus from the annotation |
| Protein/model ID | Exact transcript/protein model ID; retains `.1`, `.2`, etc. |
| Common name | One curated gene symbol, when unambiguous; see audit for alternatives |
| Model size | Transcript model's genomic span, including introns |
| Gene size | Parent gene's genomic span, including introns |
| CDS size | Sum of the annotated coding segments for that model |
| Protein size | Amino-acid count excluding a terminal stop symbol |
| Mol. wt. | Theoretical molecular weight in kDa |
| pI | Predicted pH of zero net protein charge |

If an ID is unmatched, check organism, annotation release and identifier system first. A UniProt or RefSeq accession will not necessarily equal an MSU model ID. Use a verified cross-reference or sequence alignment when identifier systems differ; do not remove suffixes and randomly choose a model. RAP/MSU conversion is available at [RAP-DB ID converter](https://rapdb.dna.naro.go.jp/converter/).
